In [1]:
from pathlib import Path

ML_ROOT = Path(r"C:\Users\menna\Downloads\Smart-Agricultural-Water-Management-main (1)\Smart-Agricultural-Water-Management-main\machine_learning")
SRC = ML_ROOT / "src"

# 1) نسخة احتياطية من الملف القديم
mt = SRC / "model_training.py"
legacy = SRC / "model_training_legacy.py"
if mt.exists() and not legacy.exists() and "label_gate" not in mt.read_text(encoding="utf-8"):
    legacy.write_text(mt.read_text(encoding="utf-8"), encoding="utf-8")

# 2) البوابة
(SRC / "label_gate.py").write_text(r'''from pathlib import Path
import pandas as pd

TARGET = "total_irrigation_m3_per_feddan"


def labels_available(root):
    p = Path(root) / "data" / "processed" / "features_engineered.csv"
    if not p.is_file():
        return False, f"Missing Phase 4 output: {p}"
    df = pd.read_csv(p)
    if TARGET not in df.columns or df[TARGET].notna().sum() == 0:
        return False, (f"No verified field-measured irrigation labels "
                       f"(column {TARGET} absent or empty). Rows: {len(df)}.")
    return True, f"{int(df[TARGET].notna().sum())} labelled rows."


def check(root, phase, name):
    ok, msg = labels_available(root)
    if not ok:
        rep = Path(root) / "reports"
        rep.mkdir(parents=True, exist_ok=True)
        (rep / f"{name}_status.md").write_text(
            f"# {phase}\n\nStatus: BLOCKED\n\n{msg}\n\nNo model artifacts were created.\n",
            encoding="utf-8")
        print(f"{phase}: BLOCKED - {msg}")
    return ok
''', encoding="utf-8")

# 3) الملفات الأربعة
(SRC / "model_training.py").write_text(r'''from src.label_gate import check


def run(root):
    if not check(root, "Phase 5 - Model Training", "phase5_model_training"):
        return False
    from src.model_training_legacy import run as legacy_run
    legacy_run(root)
    return True
''', encoding="utf-8")

for fname, phase, name in [
    ("hyperparameter_tuning.py", "Phase 6 - Hyperparameter Tuning", "phase6_hyperparameter_tuning"),
    ("evaluate.py", "Phase 7 - Model Evaluation", "phase7_model_evaluation"),
    ("explainability.py", "Phase 8 - Explainable AI", "phase8_explainability"),
]:
    (SRC / fname).write_text(f'''from src.label_gate import check


def run(root):
    if not check(root, "{phase}", "{name}"):
        return False
    raise NotImplementedError("Verified labels found: implement this phase on real labels.")
''', encoding="utf-8")

print(sorted(p.name for p in SRC.glob("*.py")))

['__init__.py', 'data_cleaning.py', 'data_loader.py', 'eda_analysis.py', 'evaluate.py', 'explainability.py', 'feature_engineering.py', 'hyperparameter_tuning.py', 'label_gate.py', 'model_training.py', 'model_training_legacy.py']
